# Phase 4b: Encoder Comparison

Trains the flood U-Net with three encoders (`resnet18`, `resnet34`, `efficientnet-b2`) and three seeds each, using exactly the setup of `unet_training.ipynb`: one-cycle schedule, BCE + Dice, same data, same epochs. Only the encoder and the seed change. Each run keeps its best-val-IoU checkpoint, tunes the threshold on val, and is scored on:

1. **Gabrielle test set (threshold labels):** how well the model reproduces the labelling rule.
2. **Idai / Beira test set (Copernicus EMS labels):** how well it detects real flooding in an unseen place. Skipped until the Idai patches exist.

The -3.5 dB VV-change rule and an Otsu threshold on `post_vv` are scored on both sets as baselines.

**Before running:**
- `dataset_patches/` must be in this repository's `data/` folder, as for `unet_training.ipynb`. Run the notebook from `models/`.
- Nine runs take about 6-7 hours on a GTX 1650. Results are saved after every run to `outputs/comparison/results.json`, and finished runs are skipped, so the notebook can be stopped and re-run.
- Once the Idai patches are ready, set `EVAL_ONLY = True` and re-run to score the saved checkpoints on Idai without retraining.

**Idai patch format** (`data/idai_patches/`): `images/*.npy` are float32, 4 x 256 x 256, bands `pre_vv, pre_vh, post_vv, post_vh`, normalised with the *Gabrielle* statistics in `dataset_info.json`. `masks/*.npy` are uint8 with matching names, 1 = EMS flood extent, permanent water = 0. Patches should lie entirely inside the EMS area of interest.

In [1]:
import glob
import json
import os
import random
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

# -------------------------------------------------------------------------
# 1. Configuration (everything except ENCODERS and SEEDS matches unet_training.ipynb)
# -------------------------------------------------------------------------
DATA_DIR = os.path.join('..', 'data', 'dataset_patches')
IDAI_DIR = os.path.join('..', 'data', 'idai_patches')  # Optional; see the format above
OUT_DIR = os.path.join('outputs', 'comparison')
RESULTS_PATH = os.path.join(OUT_DIR, 'results.json')
os.makedirs(OUT_DIR, exist_ok=True)

EVAL_ONLY = False             # True: re-score saved checkpoints only, no training

ENCODERS = ['resnet18', 'resnet34', 'efficientnet-b2']
SEEDS = [42, 43, 44]

ENCODER_WEIGHTS = 'imagenet'
BATCH_SIZE = 16
EPOCHS = 60
MAX_LR = 1e-3
WEIGHT_DECAY = 1e-4
THRESHOLD = 0.5                               # Used during training; re-tuned on val per run
THRESHOLD_GRID = np.arange(0.20, 0.81, 0.05)
RULE_DB = -3.5                                # Label rule: post_vv - pre_vv < -3.5 dB

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'
print(f"Using device: {device}")

with open(os.path.join(DATA_DIR, 'dataset_info.json')) as f:
    dataset_info = json.load(f)
BANDS = dataset_info['input_bands']
IN_CHANNELS = len(BANDS)
MEAN = np.array(dataset_info['normalisation']['mean'], dtype=np.float32)[:, None, None]
STD = np.array(dataset_info['normalisation']['std'], dtype=np.float32)[:, None, None]

c:\Users\Rodolfo\anaconda3\envs\geog761\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
# -------------------------------------------------------------------------
# 2. Dataset (same as unet_training.ipynb)
# -------------------------------------------------------------------------
class FloodPatchDataset(Dataset):
    def __init__(self, split_dir, augment=False):
        self.image_paths = sorted(glob.glob(os.path.join(split_dir, 'images', '*.npy')))
        self.mask_paths = sorted(glob.glob(os.path.join(split_dir, 'masks', '*.npy')))
        if not self.image_paths:
            raise FileNotFoundError(f"No patches found in '{split_dir}'")
        if [os.path.basename(p) for p in self.image_paths] != [os.path.basename(p) for p in self.mask_paths]:
            raise ValueError(f"Images and masks do not match in '{split_dir}'")
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.load(self.image_paths[idx])
        mask = np.load(self.mask_paths[idx])[None]
        if self.augment:
            if random.random() < 0.5:
                image, mask = image[:, :, ::-1], mask[:, :, ::-1]
            if random.random() < 0.5:
                image, mask = image[:, ::-1, :], mask[:, ::-1, :]
            k = random.randint(0, 3)
            image, mask = np.rot90(image, k, axes=(1, 2)), np.rot90(mask, k, axes=(1, 2))
        return (torch.from_numpy(np.ascontiguousarray(image)).float(),
                torch.from_numpy(np.ascontiguousarray(mask)).float())


def make_loader(dataset, shuffle):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
                      pin_memory=(device.type == 'cuda'))


train_ds = FloodPatchDataset(os.path.join(DATA_DIR, 'train'), augment=True)
val_ds = FloodPatchDataset(os.path.join(DATA_DIR, 'val'))
test_sets = {'gabrielle_test': FloodPatchDataset(os.path.join(DATA_DIR, 'test'))}
if os.path.isdir(IDAI_DIR):
    test_sets['idai_ems'] = FloodPatchDataset(IDAI_DIR)

print(f"Train / val patches: {len(train_ds)} / {len(val_ds)}")
for name, ds in test_sets.items():
    print(f"Test set '{name}': {len(ds)} patches")
if 'idai_ems' not in test_sets:
    print(f"No Idai test set at '{IDAI_DIR}' - scoring Gabrielle only")

Train / val patches: 623 / 141
Test set 'gabrielle_test': 129 patches
No Idai test set at '..\data\idai_patches' - scoring Gabrielle only


In [3]:
# -------------------------------------------------------------------------
# 3. Loss & Metrics (same as unet_training.ipynb)
# -------------------------------------------------------------------------
bce_loss = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)

def criterion(logits, masks):
    return bce_loss(logits, masks) + dice_loss(logits, masks)

def flood_metrics(tp, fp, fn):
    """Metrics for the flooded class, from pixel counts over a whole split."""
    eps = 1e-7
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    return {
        'iou': float(tp / (tp + fp + fn + eps)),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(2 * precision * recall / (precision + recall + eps)),
    }

def counts(pred, truth):
    """True-positive, false-positive and false-negative pixel counts."""
    return (pred & truth).sum().item(), (pred & ~truth).sum().item(), (~pred & truth).sum().item()

In [4]:
# -------------------------------------------------------------------------
# 4. Training & Evaluation Functions
# -------------------------------------------------------------------------
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)  # Also seeds the train loader's shuffle order


def build_model(encoder, weights=ENCODER_WEIGHTS):
    return smp.Unet(encoder_name=encoder, encoder_weights=weights,
                    in_channels=IN_CHANNELS, classes=1).to(device)


def run_epoch(model, loader, optimizer=None, scheduler=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    total_loss, tp, fp, fn = 0.0, 0, 0, 0
    with torch.set_grad_enabled(train):
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(images)
            loss = criterion(logits.float(), masks)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
            total_loss += loss.item() * images.size(0)
            c = counts(torch.sigmoid(logits) > THRESHOLD, masks > 0.5)
            tp, fp, fn = tp + c[0], fp + c[1], fn + c[2]
    return {'loss': total_loss / len(loader.dataset), **flood_metrics(tp, fp, fn)}


def train_run(encoder, seed, ckpt_path):
    """Train one encoder/seed with the one-cycle schedule; keep the best-val-IoU checkpoint."""
    seed_everything(seed)
    train_loader, val_loader = make_loader(train_ds, shuffle=True), make_loader(val_ds, shuffle=False)
    model = build_model(encoder)
    optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=MAX_LR, epochs=EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.3)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    history, best_val_iou, best_epoch = [], -1.0, 0
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_stats = run_epoch(model, train_loader, optimizer, scheduler, scaler)
        val_stats = run_epoch(model, val_loader)
        history.append({'epoch': epoch, 'train': train_stats, 'val': val_stats})
        saved = ''
        if val_stats['iou'] > best_val_iou:
            best_val_iou, best_epoch = val_stats['iou'], epoch
            torch.save({'model_state': model.state_dict(), 'encoder': encoder, 'seed': seed,
                        'in_channels': IN_CHANNELS, 'input_bands': BANDS,
                        'normalisation': dataset_info['normalisation'], 'threshold': THRESHOLD,
                        'epoch': epoch, 'val_iou': best_val_iou}, ckpt_path)
            saved = '  <- saved'
        print(f"  [{encoder} s{seed}] epoch {epoch:2d}/{EPOCHS} ({time.time() - t0:.0f}s) | "
              f"train loss {train_stats['loss']:.4f} IoU {train_stats['iou']:.3f} | "
              f"val loss {val_stats['loss']:.4f} IoU {val_stats['iou']:.3f}{saved}")
    return history, best_epoch, best_val_iou


@torch.no_grad()
def predict_probs(model, dataset):
    model.eval()
    probs, truth = [], []
    for images, masks in make_loader(dataset, shuffle=False):
        with torch.autocast(device_type=device.type, enabled=use_amp):
            logits = model(images.to(device))
        probs.append(torch.sigmoid(logits.float()).cpu().flatten())
        truth.append(masks.flatten() > 0.5)
    return torch.cat(probs), torch.cat(truth)


def tune_threshold(val_probs, val_truth):
    """Threshold with the best flood IoU on val (the test sets are not used)."""
    best_t, best_iou = THRESHOLD, -1.0
    for t in THRESHOLD_GRID:
        iou = flood_metrics(*counts(val_probs > t, val_truth))['iou']
        if iou > best_iou:
            best_t, best_iou = float(t), iou
    return best_t, best_iou


def evaluate_model(model, threshold):
    out = {}
    for name, dataset in test_sets.items():
        probs, truth = predict_probs(model, dataset)
        out[name] = flood_metrics(*counts(probs > threshold, truth))
    return out

In [5]:
# -------------------------------------------------------------------------
# 5. Train & Score Every Encoder/Seed
# -------------------------------------------------------------------------
# Results are saved after every run; runs already in results.json are not retrained,
# but every run is re-scored, so new test sets (e.g. Idai) are picked up.
results = []
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)

for encoder in ENCODERS:
    for seed in SEEDS:
        run_id = f"{encoder}_seed{seed}"
        ckpt_path = os.path.join(OUT_DIR, f"{run_id}.pt")
        run = next((r for r in results if r['run'] == run_id), None)

        if run is None:
            if EVAL_ONLY:
                print(f"{run_id}: not trained yet, skipped")
                continue
            print(f"\n=== Training {run_id} ===")
            t0 = time.time()
            history, best_epoch, best_val_iou = train_run(encoder, seed, ckpt_path)
            run = {'run': run_id, 'encoder': encoder, 'seed': seed, 'best_epoch': best_epoch,
                   'val_iou_at_0.5': best_val_iou, 'train_minutes': (time.time() - t0) / 60,
                   'history': history}
            results.append(run)

        # Score the run's best checkpoint on every available test set
        ckpt = torch.load(ckpt_path, map_location=device)
        model = build_model(encoder, weights=None)
        model.load_state_dict(ckpt['model_state'])
        threshold, val_iou = tune_threshold(*predict_probs(model, val_ds))
        ckpt['threshold'] = threshold
        torch.save(ckpt, ckpt_path)
        run.update({'threshold': threshold, 'val_iou_tuned': val_iou,
                    'test': evaluate_model(model, threshold)})
        with open(RESULTS_PATH, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"{run_id}: threshold {threshold:.2f}, " +
              ', '.join(f"{n} IoU {m['iou']:.3f}" for n, m in run['test'].items()))
        del model
        torch.cuda.empty_cache()


=== Training resnet18_seed42 ===
  [resnet18 s42] epoch  1/60 (30s) | train loss 1.7962 IoU 0.018 | val loss 1.6839 IoU 0.022  <- saved
  [resnet18 s42] epoch  2/60 (29s) | train loss 1.6188 IoU 0.036 | val loss 1.6849 IoU 0.052  <- saved
  [resnet18 s42] epoch  3/60 (30s) | train loss 1.4619 IoU 0.172 | val loss 1.3887 IoU 0.260  <- saved
  [resnet18 s42] epoch  4/60 (30s) | train loss 1.3129 IoU 0.394 | val loss 1.3152 IoU 0.387  <- saved
  [resnet18 s42] epoch  5/60 (29s) | train loss 1.1848 IoU 0.582 | val loss 1.1632 IoU 0.503  <- saved
  [resnet18 s42] epoch  6/60 (29s) | train loss 1.0889 IoU 0.598 | val loss 1.0698 IoU 0.552  <- saved
  [resnet18 s42] epoch  7/60 (29s) | train loss 1.0009 IoU 0.603 | val loss 0.9784 IoU 0.615  <- saved
  [resnet18 s42] epoch  8/60 (29s) | train loss 0.8759 IoU 0.707 | val loss 0.8512 IoU 0.709  <- saved
  [resnet18 s42] epoch  9/60 (30s) | train loss 0.7794 IoU 0.680 | val loss 0.7674 IoU 0.643
  [resnet18 s42] epoch 10/60 (30s) | train loss 0

In [6]:
# -------------------------------------------------------------------------
# 6. Baselines: the label rule and an Otsu threshold on post_vv (both in dB)
# -------------------------------------------------------------------------
def load_db(dataset):
    """All patches of a dataset in dB, (N, C, H, W), plus boolean masks (N, H, W)."""
    images = np.stack([np.load(p) * STD + MEAN for p in dataset.image_paths])
    masks = np.stack([np.load(p).astype(bool) for p in dataset.mask_paths])
    return images, masks


def otsu_threshold(values, bins=256):
    """Threshold that best separates a histogram into two classes (Otsu's method)."""
    hist, edges = np.histogram(values, bins=bins)
    centres = (edges[:-1] + edges[1:]) / 2
    w0 = np.cumsum(hist)
    w1 = w0[-1] - w0
    m0 = np.cumsum(hist * centres) / np.maximum(w0, 1)
    m1 = ((hist * centres).sum() - np.cumsum(hist * centres)) / np.maximum(w1, 1)
    return float(centres[np.argmax(w0 * w1 * (m0 - m1) ** 2)])


pre_idx, post_idx = BANDS.index('pre_vv'), BANDS.index('post_vv')
baselines = {}
for name, dataset in test_sets.items():
    images, truth = load_db(dataset)
    rule = (images[:, post_idx] - images[:, pre_idx]) < RULE_DB
    # Otsu is fitted on that test scene's own post_vv histogram (unsupervised, no labels used)
    t = otsu_threshold(images[:, post_idx].ravel())
    otsu = images[:, post_idx] < t
    baselines[name] = {
        'rule_vv_change': flood_metrics(*counts(rule, truth)),
        'otsu_post_vv': {**flood_metrics(*counts(otsu, truth)), 'threshold_db': t},
    }
    print(f"{name}: Otsu threshold on post_vv = {t:.2f} dB")

with open(os.path.join(OUT_DIR, 'baselines.json'), 'w') as f:
    json.dump(baselines, f, indent=2)

gabrielle_test: Otsu threshold on post_vv = -16.33 dB


In [7]:
# -------------------------------------------------------------------------
# 7. Summary: mean ± std over seeds, with the baselines
# -------------------------------------------------------------------------
for name in test_sets:
    print(f"\n{name}  (U-Nets: mean ± std over seeds)")
    print(f"  {'model':18}{'n':>3}  {'IoU':>15}  {'precision':>15}  {'recall':>15}")
    for enc in ENCODERS:
        runs = [r for r in results if r['encoder'] == enc and name in r.get('test', {})]
        if not runs:
            continue
        cells = []
        for k in ('iou', 'precision', 'recall'):
            v = np.array([r['test'][name][k] for r in runs])
            cells.append(f"{v.mean():.3f} ± {v.std(ddof=1) if len(v) > 1 else 0:.3f}")
        print(f"  {enc:18}{len(runs):>3}  " + '  '.join(f"{c:>15}" for c in cells))
    for b, label in (('rule_vv_change', 'rule (VV < -3.5dB)'), ('otsu_post_vv', 'Otsu post_vv')):
        m = baselines[name][b]
        print(f"  {label:18}{'-':>3}  " + '  '.join(f"{m[k]:>15.3f}" for k in ('iou', 'precision', 'recall')))


gabrielle_test  (U-Nets: mean ± std over seeds)
  model               n              IoU        precision           recall
  resnet18            3    0.954 ± 0.002    0.975 ± 0.001    0.978 ± 0.002
  resnet34            3    0.952 ± 0.002    0.977 ± 0.001    0.974 ± 0.001
  efficientnet-b2     3    0.956 ± 0.001    0.976 ± 0.001    0.979 ± 0.001
  rule (VV < -3.5dB)  -            0.570            0.570            1.000
  Otsu post_vv        -            0.050            0.051            0.840


## Notes

- **Outputs are saved to `models/outputs/comparison/`:** one checkpoint per run (`<encoder>_seed<n>.pt`, with the val-tuned threshold), `results.json` (per-run metrics and full training histories) and `baselines.json`. They are not tracked by git.
- **What the Gabrielle numbers mean.** The labels are a -3.5 dB VV-change threshold outside JRC permanent water, so every encoder is expected to score about the same there; differences between them mostly reflect how closely each copies that rule. The Otsu baseline is low on this set because it also marks the sea and rivers, which these labels exclude.
- **What the Idai numbers mean.** The EMS flood extent is independent of the inputs, so this is the test of real flood detection, in a place none of the models has seen. Expect lower scores than on Gabrielle; differences between encoders, and against the two baselines, are the ones worth discussing.
- **Seeds.** Std is over three seeds, so treat a gap between encoders smaller than about two standard deviations as no real difference.